In [23]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_FiLMSEModel, load_LoRASEModel, load_FiLMLoRASEModel, load_HyperNetworkSEModel, generate_files_with_nucleus
from models_graph import HarmonicGraphEncoder
import torch
import numpy as np
import pickle
from tqdm import tqdm
from GridMLM_tokenizers import CSGridMLMTokenizer
from graph_utils import chord_id_features, get_graph_embeddings_from_string_with_model, get_bilstm_embeddings_from_string_with_model, get_token_bilstm_embeddings_from_string_with_model, make_graph_ready_for_token_ids

In [24]:
print(chord_id_features.keys())

dict_keys([7, 36, 65, 94, 123, 152, 181, 210, 239, 268, 297, 326, 8, 37, 66, 95, 124, 153, 182, 211, 240, 269, 298, 327, 9, 38, 67, 96, 125, 154, 183, 212, 241, 270, 299, 328, 10, 39, 68, 97, 126, 155, 184, 213, 242, 271, 300, 329, 11, 40, 69, 98, 127, 156, 185, 214, 243, 272, 301, 330, 12, 41, 70, 99, 128, 157, 186, 215, 244, 273, 302, 331, 13, 42, 71, 100, 129, 158, 187, 216, 245, 274, 303, 332, 14, 43, 72, 101, 130, 159, 188, 217, 246, 275, 304, 333, 15, 44, 73, 102, 131, 160, 189, 218, 247, 276, 305, 334, 16, 45, 74, 103, 132, 161, 190, 219, 248, 277, 306, 335, 17, 46, 75, 104, 133, 162, 191, 220, 249, 278, 307, 336, 18, 47, 76, 105, 134, 163, 192, 221, 250, 279, 308, 337, 19, 48, 77, 106, 135, 164, 193, 222, 251, 280, 309, 338, 20, 49, 78, 107, 136, 165, 194, 223, 252, 281, 310, 339, 21, 50, 79, 108, 137, 166, 195, 224, 253, 282, 311, 340, 22, 51, 80, 109, 138, 167, 196, 225, 254, 283, 312, 341, 23, 52, 81, 110, 139, 168, 197, 226, 255, 284, 313, 342, 24, 53, 82, 111, 140, 169, 19

In [25]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [26]:
device_name = 'cuda:0'
device = torch.device(device_name)

# graph_model_FiLM_path = 'saved_models/FiLM/graph/graph_model.pt'
# transformer_graph_FiLM_path = 'saved_models/FiLM/graph/transformer_model.pt'

# graph_model_LoRA_path = 'saved_models/LoRA/graph/graph_model_jnhw.pt'
# transformer_graph_LoRA_path = 'saved_models/LoRA/graph/transformer_model_jnhw.pt'
graph_model_LoRA_path = 'saved_models/LoRA/graph/graph_model_contra_jnhw.pt'
transformer_graph_LoRA_path = 'saved_models/LoRA/graph/transformer_model_contra_jnhw.pt'

# graph_model_FiLMLoRA_path = 'saved_models/FiLMLoRA/graph/graph_model.pt'
# transformer_graph_FiLMLoRA_path = 'saved_models/FiLMLoRA/graph/transformer_model.pt'

# graph_model_HyperNetwork_path = 'saved_models/HyperNetwork/graph/graph_model.pt'
# transformer_graph_HyperNetwork_path = 'saved_models/HyperNetwork/graph/transformer_model.pt'

bilstm_model_path = 'saved_models/LoRA/bilstm/bilstm_model_contra_jnhw.pt'
transformer_bilstm_path = 'saved_models/LoRA/bilstm/transformer_model_contra_jnhw.pt'

# token_bilstm_model_path = 'saved_models/HyperNetwork/token_bilstm/bilstm_model.pt'
# transformer_token_bilstm_path = 'saved_models/HyperNetwork/token_bilstm/transformer_model.pt'
token_bilstm_model_path = 'saved_models/LoRA/token_bilstm/bilstm_model_contra_jnhw.pt'
transformer_token_bilstm_path = 'saved_models/LoRA/token_bilstm/transformer_model_contra_jnhw.pt'

In [27]:
# graph_model_FiLM = load_GraphModel(graph_model_FiLM_path, device)
# transformer_graph_FiLM = load_FiLMSEModel(
#     tokenizer,
#     device,
#     checkpoint_path=transformer_graph_FiLM_path
# )

# graph_model_LoRA = graph_model = HarmonicGraphEncoder(participation_edge_dim=5)
# graph_model_LoRA.to(device)
graph_model_LoRA = load_GraphModel(graph_model_LoRA_path, device)
transformer_graph_LoRA = load_LoRASEModel(
    tokenizer,
    device,
    checkpoint_path=transformer_graph_LoRA_path
)

# graph_model_FiLMLoRA = load_GraphModel(graph_model_FiLMLoRA_path, device)
# transformer_graph_FiLMLoRA = load_FiLMLoRASEModel(
#     tokenizer,
#     device,
#     checkpoint_path=transformer_graph_FiLMLoRA_path
# )
# graph_model_HyperNetwork = load_GraphModel(graph_model_HyperNetwork_path, device)
# transformer_graph_HyperNetwork = load_HyperNetworkSEModel(
#     tokenizer,
#     device,
#     checkpoint_path=transformer_graph_HyperNetwork_path
# )

bilstm_model = load_BiLSTMModel(bilstm_model_path, device)
transformer_bilstm = load_LoRASEModel(
    tokenizer,
    device,
    checkpoint_path=transformer_bilstm_path
)

token_bilstm_model = load_TokenBiLSTMModel(token_bilstm_model_path, tokenizer, device)
transformer_token_bilstm = load_LoRASEModel(
    tokenizer,
    device,
    checkpoint_path=transformer_token_bilstm_path
)

In [28]:
# graph_model_FiLM.eval()
# transformer_graph_FiLM.eval()

graph_model_LoRA.eval()
transformer_graph_LoRA.eval()

# graph_model_FiLMLoRA.eval()
# transformer_graph_FiLMLoRA.eval()

# graph_model_HyperNetwork.eval()
# transformer_graph_HyperNetwork.eval()

bilstm_model.eval()
transformer_bilstm.eval()

token_bilstm_model.eval()
transformer_token_bilstm.eval()

LoRASEModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (layers): ModuleList(
    (0-7): 8 x TransformerBlockWithAttnLoRA(
      (attn): MultiHeadAttentionWithAttnLoRA(
        (q_proj): Linear(in_features=512, out_features=512, bias=True)
        (k_proj): Linear(in_features=512, out_features=512, bias=True)
        (v_proj): Linear(in_features=512, out_features=512, bias=True)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
        (q_lora): ModuleList(
          (0-7): 8 x HyperLoRA(
            (lora_A): Linear(in_features=512, out_features=32, bias=True)
            (lora_B): Linear(in_features=512, out_features=32, bias=True)
          )
        )
        (k_lora): ModuleList(
          (0-7): 8 x HyperLoRA(
            (lora_A): Linear(in_features=512, out_features=32, bias=True)
            (lora_B): Linear(in_features=512, out_features=32, bias=True)
          )
        )
        (v

In [29]:
datasets = {
    'gjt': {'path': 'data/gjt_test.pkl', 'dataset': None},
    # 'hook': {'path': 'data/hook_test.pkl', 'dataset': None},
    'nott': {'path': 'data/nott_test.pkl', 'dataset': None},
    # 'wiki': {'path': 'data/wiki_test.pkl', 'dataset': None}
}

for k, v in datasets.items():
    print(f'loading {k}')
    with open(v['path'], 'rb') as f:
        d = pickle.load(f)
    v['dataset'] = d

loading gjt
loading nott


In [30]:
# in_seq = 'b_A#:7_@2;A:min6_@2'

# in_seq = 'b_E:7_@2;A:min_@2'
# in_seq = 'b_G:7_@2;C:maj_@2'
# in_seq = 'b_B:hdim7_@2;A#:7_@2'
# in_seq = 'b_F:7_@2;E:7_@2'
# in_seq = 'b_C:7_@2;B:7_@2b_A#:7_@2;A:min6_@2'
# in_seq = 'b_A:min_@4'
# in_seq = 'b_D#:dim7_@4'

# in_seq = 'b_F#:7_@2;B:7_@2b_E:7_@2;A#:7_@2;'
# in_seq = 'b_A#:7_@2;A:min6_@2'
# in_seq = 'b_F:min_@2;A#:min_@2'
# in_seq = 'b_E:7_@2;A:7_@2b_F:7_@2;A#:7_@2'
# in_seq = 'b_E:min6_@2;A:7_@2b_F:min6_@2;A#:7_@2'
# in_seq = 'G:maj;D:maj;G#:maj;D#:maj'
in_seq = 'b_C#:hdim7_@4b_D:hdim7_@4'

In [31]:
# y_graph_FiLM = get_graph_embeddings_from_string_with_model(in_seq, graph_model_FiLM)
y_graph_LoRA = get_graph_embeddings_from_string_with_model(in_seq, graph_model_LoRA)
# y_graph_FiLMLoRA = get_graph_embeddings_from_string_with_model(in_seq, graph_model_FiLMLoRA)
# y_graph_HyperNetwork = get_graph_embeddings_from_string_with_model(in_seq, graph_model_HyperNetwork)

y_bilstm = get_bilstm_embeddings_from_string_with_model(in_seq, bilstm_model)
y_token_bilstm = get_token_bilstm_embeddings_from_string_with_model(in_seq, token_bilstm_model)

C#:hdim7 in vocab as: 49
D:hdim7 in vocab as: 78
C#:hdim7 in vocab as: 49
D:hdim7 in vocab as: 78
C#:hdim7 in vocab as: 49
D:hdim7 in vocab as: 78


In [32]:
# print(y_bilstm.shape)
# print(y_token_bilstm.shape)

In [33]:
# input_f_path = '/mnt/ssd2/maximos/data/gjt_melodies/gjt_CA_test/Autumn_Leaves.mxl'
# input_f_path = '/mnt/ssd2/maximos/data/gjt_melodies/gjt_CA_test/So_What.mxl'

# input_f_path = '/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/So_What.mxl'
input_f_path = '/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/Autumn_Leaves.mxl'
# input_f_path = '/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/Off_Minor.mxl'

piece_name = input_f_path.split('/')[-1].split('.')[0]
print(piece_name)

Autumn_Leaves


In [34]:
gen_out = generate_files_with_nucleus(
    transformer_graph_LoRA,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_no',
    guidance_vec = None,
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])
for i in gen_out['decoded_positions_order']:
    print(i, '-', gen_out['gen_output_tokens'][i])

['<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'D:min7', 'D:min7', 'D:min7', 'D:min7', '<bar>', 'G:7', 'G:7', 'G:7', 'G:7', '<bar>', 'C:maj7', 'C:maj7', 'C:maj7', 'C:maj7', '<bar>', 'D:min7', 'D:min7', 'D:min7', 'D:min7', '<bar>', 'B:hdim7', 'B:hdim7', 'B:hdim7', 'B:hdim7', '<bar>', 'B:hdim7', 'B:hdim7', 'E:7', 'E:7', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'B:hdim7', 'B:hdim7', 'E:7(b9)', 'E:7(b9)', '<bar>', 'A:min7', 'A:min7', 'F:7', 'F:7', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'B:hdim7', 'B:hdim7', 'B:hdim7', 'B:hdim7', '<bar>', 'B:hdim7', 'B:hdim7', 'B:hdim7', 'E:7(b9)', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'A:min', 'A:min', 'B:hdim7', 'B:hdim7']
39 - A:min
38 - A:min
37 - A:min
36 - A:min
34 - E:7
33 - E:7
41 - A:min
42 - A:min
44 - A:min
43 - A:min
46 - B:hdim7
47 - B:hdim7
26 - B:hdim7
27 - B:hdim7
28 - B:hdim7
29 - B:hdim7
66 - B:hdim7
67 - B:hdim7
74 - A:min
73 - A:min
72 - A:mi

In [35]:
# gen_out = generate_files_with_nucleus(
#     transformer_graph_FiLM,
#     tokenizer,
#     input_f_path=input_f_path,
#     mxl_folder_out=None,
#     midi_folder_out='midi_tests',
#     name_suffix=f'{piece_name}_graph_FiLM',
#     guidance_vec = y_graph_FiLM,
#     use_constraints=False,
#     intertwine_bar_info=True,
#     normalize_tonality=True,
#     temperature=1.0,
#     p=0.9,
#     unmasking_order='certain',
#     create_gen=True,
#     create_real=False
# )
# print(gen_out['gen_output_tokens'])

In [36]:
gen_out = generate_files_with_nucleus(
    transformer_graph_LoRA,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_graph_LoRA',
    guidance_vec = y_graph_LoRA,
    num_guidance_steps=16,
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])
for i in gen_out['decoded_positions_order']:
    print(i, '-', gen_out['gen_output_tokens'][i])

stopping guidance - decoded: [39, 38, 37, 36, 6, 7, 8, 9, 63, 62, 64, 61, 79, 78, 16, 17, 18]
['<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'D:hdim7', 'D:hdim7', 'D:hdim7', 'D:hdim7', '<bar>', 'G:7', 'G:7', 'G:7', 'G:7', '<bar>', 'E:hdim7', 'E:hdim7', 'E:hdim7', 'C:maj7', '<bar>', 'D:min7', 'D:min7', 'D:min7', 'D:min7', '<bar>', 'B:hdim7', 'B:hdim7', 'B:hdim7', 'B:hdim7', '<bar>', 'B:hdim7', 'B:hdim7', 'E:7', 'E:7', '<bar>', 'E:7(b13)', 'E:7(b13)', 'E:7(b13)', 'E:7(b13)', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'B:hdim7', 'B:hdim7', 'E:7', 'E:7', '<bar>', 'A:min7', 'A:min7', 'A:min7', 'A:min7', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'C#:hdim7', 'C#:hdim7', 'C#:hdim7', 'C#:hdim7', '<bar>', 'B:hdim7', 'B:hdim7', 'E:7', 'E:7', '<bar>', 'C:maj7', 'C:maj7', 'C:maj7', 'C:maj7', '<bar>', 'B:hdim7', 'B:hdim7', 'E:7(b13)', 'E:7(b13)']
39 - E:7(b13)
38 - E:7(b13)
37 - E:7(b13)
36 - E:7(b13)
6 - D:hdim7
7 - D:hdim7
8 - D:hdim7
9 - D:hdim7
63 - C#:hdim7
62 - C

In [37]:
# gen_out = generate_files_with_nucleus(
#     transformer_graph_FiLMLoRA,
#     tokenizer,
#     input_f_path=input_f_path,
#     mxl_folder_out=None,
#     midi_folder_out='midi_tests',
#     name_suffix=f'{piece_name}_graph_FiLMLoRA',
#     guidance_vec = y_graph_FiLMLoRA,
#     use_constraints=False,
#     intertwine_bar_info=True,
#     normalize_tonality=True,
#     temperature=1.0,
#     p=0.9,
#     unmasking_order='certain',
#     create_gen=True,
#     create_real=False
# )
# print(gen_out['gen_output_tokens'])

In [38]:
# gen_out = generate_files_with_nucleus(
#     transformer_graph_HyperNetwork,
#     tokenizer,
#     input_f_path=input_f_path,
#     mxl_folder_out=None,
#     midi_folder_out='midi_tests',
#     name_suffix=f'{piece_name}_graph_HyperNetwork',
#     guidance_vec = y_graph_HyperNetwork,
#     use_constraints=False,
#     intertwine_bar_info=True,
#     normalize_tonality=True,
#     temperature=1.0,
#     p=0.9,
#     unmasking_order='certain',
#     create_gen=True,
#     create_real=False
# )
# print(gen_out['gen_output_tokens'])

In [39]:
gen_out = generate_files_with_nucleus(
    transformer_bilstm,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_bilstm',
    guidance_vec = y_bilstm,
    num_guidance_steps=16,
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])
for i in gen_out['decoded_positions_order']:
    print(i, '-', gen_out['gen_output_tokens'][i])

stopping guidance - decoded: [26, 27, 16, 17, 18, 19, 28, 29, 56, 54, 53, 52, 51, 57, 58, 59, 46]
['<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'D:min7', 'D:min7', 'D:min7', 'D:min7', '<bar>', 'G:7', 'G:7', 'G:7', 'G:7', '<bar>', 'C:maj7', 'C:maj7', 'C:maj7', 'C:maj7', '<bar>', 'D:min7', 'D:min7', 'D:min7', 'D:min7', '<bar>', 'B:hdim7', 'B:hdim7', 'B:hdim7', 'B:hdim7', '<bar>', 'B:hdim7', 'B:hdim7', 'E:7', 'E:7', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'B:hdim7', 'B:hdim7', 'E:7(b9)', 'E:7(b9)', '<bar>', 'F:7', 'F:7', 'F:7', 'F:7', '<bar>', 'F:7', 'F:7', 'F:7', 'F:7', '<bar>', 'B:hdim7', 'B:hdim7', 'B:hdim7', 'B:hdim7', '<bar>', 'B:hdim7', 'B:hdim7', 'E:7(b9)', 'E:7(b9)', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'A:min', 'A:min', 'E:7', 'E:7']
26 - B:hdim7
27 - B:hdim7
16 - C:maj7
17 - C:maj7
18 - C:maj7
19 - C:maj7
28 - B:hdim7
29 - B:hdim7
56 - F:7
54 - F:7
53 - F:7
52 - F:7
51 - F:7
57 - F:7
58 - F:7
59

In [40]:
gen_out = generate_files_with_nucleus(
    transformer_token_bilstm,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_token_bilstm',
    guidance_vec = y_token_bilstm,
    num_guidance_steps=16,
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])
for i in gen_out['decoded_positions_order']:
    print(i, '-', gen_out['gen_output_tokens'][i])

stopping guidance - decoded: [6, 7, 8, 9, 26, 27, 28, 29, 24, 22, 21, 23, 11, 12, 71, 72, 73]
['<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'D:hdim7', 'D:hdim7', 'D:hdim7', 'D:hdim7', '<bar>', 'D:hdim7', 'D:hdim7', 'G:7', 'G:7', '<bar>', 'C:maj', 'C:maj', 'C:maj', 'C:maj', '<bar>', 'D:hdim7', 'D:hdim7', 'D:hdim7', 'D:hdim7', '<bar>', 'D:hdim7', 'D:hdim7', 'D:hdim7', 'D:hdim7', '<bar>', 'E:7', 'E:7', 'E:7', 'E:7', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'B:hdim7', 'B:hdim7', 'B:hdim7', 'B:hdim7', '<bar>', 'D:min', 'D:min', 'D:min', 'D:min', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'A:min', 'A:min', 'F#:hdim7', 'F#:hdim7', '<bar>', 'D:hdim7', 'D:hdim7', 'D:hdim7', 'A:min', '<bar>', 'A:min', 'A:min', 'E:7', 'E:7']
6 - D:hdim7
7 - D:hdim7
8 - D:hdim7
9 - D:hdim7
26 - D:hdim7
27 - D:hdim7
28 - D:hdim7
29 - D:hdim7
24 - D:hdim7
22 - D:hdim7
21 - D:hdim7
23 - D:hdim7
1

In [41]:
print(gen_out.keys())

dict_keys(['gen_output_tokens', 'gen_output_token_ids', 'harmony_real_tokens', 'gen_score', 'real_score', 'guide_score', 'hidden', 'decoded_positions_order'])


In [42]:
print(gen_out['gen_output_token_ids'][0])

tensor([  6, 269, 269, 269, 269,   6,  78,  78,  78,  78,   6,  78,  78, 216,
        216,   6,   7,   7,   7,   7,   6,  78,  78,  78,  78,   6,  78,  78,
         78,  78,   6, 129, 129, 129, 129,   6, 269, 269, 269, 269,   6, 269,
        269, 269, 269,   6, 339, 339, 339, 339,   6,  66,  66,  66,  66,   6,
        269, 269, 269, 269,   6, 269, 269, 269, 269,   6, 269, 269, 194, 194,
          6,  78,  78,  78, 269,   6, 269, 269, 129, 129], device='cuda:0')


In [43]:
m = make_graph_ready_for_token_ids(gen_out['gen_output_token_ids'][0].tolist(), tokenizer)

In [44]:
m.make_graph_of_segment(0,4)
m.make_bilstm_seq_of_segment(0,4)